# Store Sales - Preprocessing

이 노트북은 **전처리와 검증용 데이터 분할**만 담당합니다.

포함:
- `train.csv`, `test.csv`, `holidays_events.csv` 로드
- 날짜형 변환
- `store_nbr`, `family` 범주형 변환
- 시간 기준 Train / Validation 분리
- `sales`, `onpromotion` 이상치 처리 방향 정리
- Target (`sales`)의 `log1p` 변환
- `holidays` 기본 전처리
  - 날짜형 변환
  - 완전 중복 제거
  - `transferred=True`인 원래 휴일 날짜 제거
  - 정렬

제외:
- 날짜 파생 feature (`year`, `month`, `dayofweek` 등)
- `onpromotion_log`, `is_promotion`
- Holiday를 실제 매장별로 매칭하는 작업
- Lag / Rolling

위 항목들은 **Feature Engineering** 단계에서 처리합니다.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display


## 1. 데이터 경로 설정

In [ ]:
cwd = Path.cwd()

if (cwd / "data").exists():
    DATA_DIR = cwd / "data"
elif (cwd.parent / "data").exists():
    DATA_DIR = cwd.parent / "data"
else:
    raise FileNotFoundError("data 폴더를 찾을 수 없습니다.")

print("DATA_DIR:", DATA_DIR)


## 2. 데이터 로드

In [ ]:
def load_data():
    train = pd.read_csv(DATA_DIR / "train.csv")
    test = pd.read_csv(DATA_DIR / "test.csv")
    holidays = pd.read_csv(DATA_DIR / "holidays_events.csv")

    return train, test, holidays


train, test, holidays = load_data()

print("train   :", train.shape)
print("test    :", test.shape)
print("holidays:", holidays.shape)


## 3. 날짜형 변환

In [ ]:
train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])
holidays["date"] = pd.to_datetime(holidays["date"])

print("train date dtype   :", train["date"].dtype)
print("test date dtype    :", test["date"].dtype)
print("holidays date dtype:", holidays["date"].dtype)


## 4. 범주형 변수 처리

In [ ]:
def convert_categories(df):
    df = df.copy()

    df["store_nbr"] = df["store_nbr"].astype("category")
    df["family"] = df["family"].astype("category")

    return df


train = convert_categories(train)
test = convert_categories(test)

print(train[["store_nbr", "family"]].dtypes)


## 5. Train / Validation Split

실제 Kaggle test가 **16일**이므로, 학습 데이터의 마지막 16일을 validation으로 사용합니다.

- Train: 2013-01-01 ~ 2017-07-30
- Validation: 2017-07-31 ~ 2017-08-15
- Test: 2017-08-16 ~ 2017-08-31


In [ ]:
def split_train_valid(train, valid_days=16):
    valid_start = train["date"].max() - pd.Timedelta(days=valid_days - 1)

    train_df = train[train["date"] < valid_start].copy()
    valid_df = train[train["date"] >= valid_start].copy()

    return train_df, valid_df


train_df, valid_df = split_train_valid(train)

print("Train:", train_df["date"].min(), "~", train_df["date"].max())
print("Valid:", valid_df["date"].min(), "~", valid_df["date"].max())
print("Test :", test["date"].min(), "~", test["date"].max())

print()
print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)
print("Test shape :", test.shape)


## 6. 이상치 처리 방향

EDA 및 전처리 분석 결과:

### `sales`
- IQR 기준으로 제거하면 정상적인 고판매량까지 너무 많이 제거됨
- 상위 극단값이 주로 `GROCERY I`, `BEVERAGES` 등 고판매 상품군에 집중됨
- 따라서 **극단값을 일괄 삭제하지 않음**

### `onpromotion`
- 높은 값이 특정 기간에 반복적으로 나타남
- 전체적으로 `sales`와 양의 상관관계가 확인됨
- 따라서 **극단값을 일괄 삭제하지 않음**

현재 단계에서는 `sales`, `onpromotion` 모두 원본 데이터를 유지합니다.


## 7. Target 변환

`sales`는 강한 right-skew를 보였고 `log1p` 적용 후 왜도가 크게 감소했습니다.

- 원본 `sales`는 보존
- 모델링용 target 후보로 `sales_log = log1p(sales)` 추가

`onpromotion_log`는 feature이므로 여기서는 만들지 않습니다.


In [ ]:
def add_target_log(df):
    df = df.copy()
    df["sales_log"] = np.log1p(df["sales"])
    return df


train_df = add_target_log(train_df)
valid_df = add_target_log(valid_df)

print("sales skew")
print("original:", train_df["sales"].skew())
print("log1p   :", train_df["sales_log"].skew())


## 8. Holiday 구조 확인

Holiday는 같은 날짜에도 `National`, `Regional`, `Local` 등이 동시에 존재할 수 있으므로  
**날짜만 기준으로 중복 제거하면 안 됩니다.**

먼저 원본 구조를 확인합니다.


In [ ]:
print("Holiday columns:")
print(holidays.columns.tolist())

print("\nType counts:")
print(holidays["type"].value_counts())

print("\nLocale counts:")
print(holidays["locale"].value_counts())

print("\nTransferred counts:")
print(holidays["transferred"].value_counts())

display(holidays.head(20))


## 9. Holiday 기본 전처리

여기서는 **Holiday 자체를 정리하는 전처리만** 수행합니다.

- `date`는 이미 `datetime`으로 변환됨
- 완전히 동일한 행만 제거
- `transferred=True`인 원래 휴일 날짜는 제거
  - 실제 이동된 휴일 날짜는 데이터의 `type="Transfer"` 행으로 별도 존재할 수 있으므로 삭제하지 않음
- 날짜/적용 범위 기준으로 정렬

아직 `train/test`에 merge하지 않습니다.  
`National / Regional / Local`을 실제 매장에 적용하려면 `stores.csv`의 `city`, `state` 정보가 필요하므로 Feature Engineering 단계에서 처리합니다.


In [ ]:
def preprocess_holidays(holidays):
    holidays = holidays.copy()

    # 완전히 동일한 행만 제거
    holidays = holidays.drop_duplicates()

    # 원래 날짜에서 실제 휴일로 적용되지 않는 transferred=True 제거
    holidays = holidays[
        holidays["transferred"] == False
    ].copy()

    # 정렬 및 index 정리
    holidays = (
        holidays
        .sort_values(["date", "locale", "locale_name"])
        .reset_index(drop=True)
    )

    return holidays


holidays = preprocess_holidays(holidays)

print("Preprocessed holidays shape:", holidays.shape)
print("\nTransferred counts after preprocessing:")
print(holidays["transferred"].value_counts())

display(holidays.head(20))


## 10. 같은 날짜에 여러 Holiday가 존재하는지 확인

같은 날짜에 여러 행이 남아 있는 것은 반드시 중복 오류가 아닙니다.  
서로 다른 지역 범위의 Holiday가 동시에 존재할 수 있기 때문입니다.


In [ ]:
holiday_date_counts = (
    holidays
    .groupby("date")
    .size()
    .sort_values(ascending=False)
)

display(holiday_date_counts.head(20))


## 11. 최종 확인

In [ ]:
print("train_df:", train_df.shape)
print("valid_df:", valid_df.shape)
print("test    :", test.shape)
print("holidays:", holidays.shape)

print("\ntrain_df columns:")
print(train_df.columns.tolist())

print("\nholidays columns:")
print(holidays.columns.tolist())


In [15]:
full_dates = pd.date_range(
    train["date"].min(),
    train["date"].max(),
    freq="D"
)

missing_dates = full_dates.difference(
    train["date"].drop_duplicates()
)

print("Missing dates:", len(missing_dates))
print(missing_dates)

Missing dates: 4
DatetimeIndex(['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25'], dtype='datetime64[us]', freq=None)


In [16]:
holidays[
    holidays["date"].isin(missing_dates)
].sort_values("date")

,date,type,locale,locale_name,description,transferred,month_day
89,2013-12-25,Holiday,National,Ecuador,Navidad,False,12-25
155,2014-12-25,Holiday,National,Ecuador,Navidad,False,12-25
208,2015-12-25,Holiday,National,Ecuador,Navidad,False,12-25
294,2016-12-25,Holiday,National,Ecuador,Navidad,False,12-25


## 다음 단계: Feature Engineering

다음 단계에서는 새로운 모델 입력 변수를 만듭니다.

### 날짜 Feature
- `year`
- `month`
- `day`
- `dayofweek`
- `is_weekend`

### Promotion Feature
- `onpromotion_log`
- `is_promotion`

### Holiday Feature
`stores.csv`를 이용해 각 Holiday가 실제로 적용되는 store를 판단합니다.

- National → 모든 매장
- Regional → 해당 `state`의 매장
- Local → 해당 `city`의 매장

이후 `date × store_nbr` 단위로 Holiday 정보를 정리해 `train/valid/test`에 merge합니다.

### 시계열 Feature
- Lag
- Rolling
- 과거 판매 0 비율 등


sales 결측치 처리

In [18]:
full_dates = pd.date_range(
    train["date"].min(),
    train["date"].max(),
    freq="D"
)

In [19]:
missing_dates = full_dates.difference(
    train["date"].drop_duplicates()
)

print("Missing dates:", len(missing_dates))
print(missing_dates)

Missing dates: 0
DatetimeIndex([], dtype='datetime64[us]', freq='D')


In [ ]:
# 빠진 크리스마스 날짜
missing_dates = pd.to_datetime([
    "2013-12-25",
    "2014-12-25",
    "2015-12-25",
    "2016-12-25"
])

# 모든 매장과 상품군
stores = train["store_nbr"].unique()
families = train["family"].unique()

# 날짜 × 매장 × 상품군의 모든 조합 생성
missing_rows = pd.MultiIndex.from_product(
    [missing_dates, stores, families],
    names=["date", "store_nbr", "family"]
).to_frame(index=False)

# 휴점일로 판단하여 0으로 복원
missing_rows["sales"] = 0.0
missing_rows["onpromotion"] = 0

# 원본에는 없는 행임을 표시
missing_rows["is_added_missing_date"] = 1

# id는 원본 Kaggle id가 없으므로 임시값
missing_rows["id"] = -1


# 기존 데이터는 실제 관측값
train["is_added_missing_date"] = 0


# 컬럼 순서 통일
missing_rows = missing_rows[
    [
        "id",
        "date",
        "store_nbr",
        "family",
        "sales",
        "onpromotion",
        "is_added_missing_date"
    ]
]


# 원본 train에 추가
train = pd.concat(
    [train, missing_rows],
    ignore_index=True
)


# 날짜 → 매장 → 상품군 순으로 정렬
train = train.sort_values(
    ["date", "store_nbr", "family"]
).reset_index(drop=True)

In [20]:
missing_dates = full_dates.difference(
    train["date"].drop_duplicates()
)

print("Missing dates:", len(missing_dates))
print(missing_dates)

Missing dates: 0
DatetimeIndex([], dtype='datetime64[us]', freq='D')
